## What可视化

2D Grand CAM可视化模块

In [1]:
from onekey_algo.datasets.image_loader import default_loader
from onekey_algo.custom.components.comp2 import show_cam_on_image
import torch
import os
import random
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.comp2 import extract, init_from_model, init_from_onekey, target_layer_mapping
from onekey_algo.utils.MultiProcess import MultiProcess
import numpy as np
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import monai
from glob import glob
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib
matplotlib.use('Agg')

samples = glob(os.path.join(r'D:\20251014-fenghexin_fuxin\L', '*.npy'))
model_root = r'D:/20251014-fenghexin_fuxin/Lmodels/CV-0/vgg19'
random.shuffle(samples)
def viz_sample(samples, thread_id):
    model, transformer, device = init_from_onekey(os.path.join(model_root, 'viz'),
                                                  force_pth=r'D:/20251014-fenghexin_fuxin/Lmodels/CV-0/vgg19\train/training-params-33.pth')
    for n, m in model.named_modules():
        print('Feature name:', n, "|| Module:", m)
    target_layer = target_layer_mapping[os.path.basename(model_root) + '_2D']
    gradcam = monai.visualize.GradCAM(nn_module=model, target_layers=target_layer)
#     return
    viz_dir = os.path.join(model_root, 'Grad-CAM')
    os.makedirs(viz_dir, exist_ok=True)
    for sample in samples:
        if os.path.exists(os.path.join(viz_dir, os.path.basename(sample))):
            continue
        img = default_loader(sample)
        num_channels = img.shape[-1]
        sample_ = transformer(img)
        sample_  = sample_.view(1, *sample_.size()).to(device)
        res_cam = gradcam(x=sample_, class_idx=None)
        fig, axes = plt.subplots(1, num_channels + 1, figsize=((num_channels + 1) * 4, 4), facecolor='white')
    #     axes[0].imshow(-res_cam[0][0].cpu(), cmap='jet')
        for i in range(num_channels):
#             print(np.sum(img[..., i]))
            axes[i].imshow(img[..., i], cmap='gray')
            axes[i].axis('off')
    #     plt.savefig(f"viz/{os.path.basename(sample).replace('.png', '_se.png')}", bbox_inches = 'tight')
    #     plt.show()
    #     plt.figure(figsize=(10, 10))
    #     plt.axis('off')
        imshow = axes[num_channels].imshow(-res_cam[0][0].cpu(),cmap='jet')
        axes[num_channels].axis('off')
        cax = fig.add_axes([0.92, 0.17, 0.02, axes[num_channels].get_position().height]) 
        plt.colorbar(imshow, cax=cax)
        plt.savefig(f'{viz_dir}/{os.path.basename(sample).replace(".npy", ".png")}', bbox_inches = 'tight')
        plt.show()
        plt.close(fig)
viz_sample(samples, thread_id=1)
# MultiProcess(func=viz_sample, samples=samples, num_process=1).run()

[2026-02-11 01:36:53 - comp2.py: 214]	INFO	Using force param files: D:/20251014-fenghexin_fuxin/Lmodels/CV-0/vgg19\train/training-params-33.pth
[2026-02-11 01:36:53 - comp2.py: 235]	INFO	模型参数：{'pretrained': False, 'model_name': 'vgg19', 'num_classes': 1, 'in_channels': 3}


Feature name:  || Module: VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(